In [1]:
# import packages
import cv2
import glob
import numpy as np
import rawpy
import rawpy.enhance
import gc
import os
import psutil


/Users/Liam/anaconda3/envs/stitch/lib/python3.11/site-packages/rawpy/enhance.py:19: UserWarning: scikit-image not found, will use OpenCV (error: No module named 'skimage')
  warnings.warn('scikit-image not found, will use OpenCV (error: ' + str(e) + ')')


In [2]:
# set params

# set filepaths
use_external_drive = True
external_raw_data_dir = "/Volumes/ECM_backup/CPL26_bubbles/2502"
home_raw_data_dir = "raw_data"
raw_data_dir = external_raw_data_dir if use_external_drive else home_raw_data_dir
output_dir = "image_outputs/"

# confidence threshold and scaling parameters
scan_conf_thresh_lowres = 1  # starting confidence threshold for low-resolution panorama
scan_conf_thresh_highres = 1 # starting confidence threshold for high-resolution panorama
scan_min_inclusion_ratio = 0.90 # minimum ratio of inliers to total matches
number_of_attempts = 6 # total number of attempts before giving up
scale_factor = 0.20  # Downsample to 20% size for feature matching


In [3]:
# Force OpenCV to run entirely on the CPU and completely disable OpenCL acceleration
# addresses an issue I was having (thanks Gemini)
cv2.ocl.setUseOpenCL(False)

# set up memory logging
process = psutil.Process(os.getpid())

def log_memory(stage):
    rss_mb = process.memory_info().rss / (1024 ** 2)
    print(f"  [Memory] {stage}: {rss_mb:.1f} MB")

In [4]:
# find all sections in the raw data folder. Assumes folder names are structured as "raw_data/<section_name>"
sampleID = sorted(glob.glob(raw_data_dir + "/*"))

# remove the leading "raw_data/" from each folder name
sampleID = [folder.replace(raw_data_dir + "/", "") for folder in sampleID]


# NOTE: Alternatively, you can mannually define a list of core sections to run as below.
#sampleID = ["ALHIC2502-102-D"]

# print summary of sample IDs (number, names)
print(f"Number of samples: {len(sampleID)}")
print(f"Sample IDs: {sampleID}")

Number of samples: 17
Sample IDs: ['ALHIC2502-101-A', 'ALHIC2502-101-B', 'ALHIC2502-101-C', 'ALHIC2502-101-D', 'ALHIC2502-102-A', 'ALHIC2502-102-B', 'ALHIC2502-102-C', 'ALHIC2502-102-D', 'ALHIC2502-103', 'ALHIC2502-24', 'ALHIC2502-33', 'ALHIC2502-33A', 'ALHIC2502-33B', 'ALHIC2502-33C', 'ALHIC2502-33D', 'ALHIC2502-47', 'ALHIC2502-86']


In [ ]:

# Loop through folders safely
for sample in sampleID:
    print("\n" + "="*50)
    print(f"Running images from: {sample}")
    print("="*50)
    log_memory("start sample")

    # 1. Locate files dynamically inside the target sample folder
    nef_paths = sorted(glob.glob(f"{raw_data_dir}/{sample}/*.NEF"))
    print(f" Found {len(nef_paths)} NEF files in folder '{sample}'.")
    
    if len(nef_paths) == 0:
        print("  [Warning] No files found. Skipping this folder.")
        continue

    def count_integrated_images(stitcher):
        try:
            return len(stitcher.cameras())
        except Exception:
            return 0

    attempt_thresholds = []
    for attempt_idx in range(1, number_of_attempts + 1):
        if attempt_idx == 1:
            low_delta = 0.0
            high_delta = 0.0
        elif attempt_idx == 2:
            low_delta = 0.10
            high_delta = 0.05
        else:
            low_delta = 0.10 + (attempt_idx - 2) * 0.05
            high_delta = 0.05 + (attempt_idx - 2) * 0.05

        attempt_thresholds.append((
            max(0.0, scan_conf_thresh_lowres - low_delta),
            max(0.0, scan_conf_thresh_highres - high_delta)
        ))

    stitched = False
    for attempt_idx, (lowres_thresh, highres_thresh) in enumerate(attempt_thresholds, start=1):
        print(f" Attempt {attempt_idx}/{number_of_attempts} with lowres={lowres_thresh:.2f}, highres={highres_thresh:.2f}")

        # Step 1: Extract low-res thumbnails to calculate alignment geometry
        print(" Step 1: Extracting low-res thumbnails to calculate alignment...")
        low_res_images = []
        for path in nef_paths:
            with rawpy.imread(path) as raw:
                rgb = raw.postprocess(use_camera_wb=True, half_size=True)

            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            h, w = bgr.shape[:2]
            target_w = int(w * scale_factor * 2)
            target_h = int(h * scale_factor * 2)
            small_img = cv2.resize(bgr, (target_w, target_h), interpolation=cv2.INTER_AREA)
            low_res_images.append(np.ascontiguousarray(small_img, dtype=np.uint8))
            del rgb, bgr, small_img

        log_memory("after low-res prep")

        # Step 2: Test alignment geometry using flat grid SCANS mode
        print(" Step 2: Aligning 2D zig-zag grid structure...")
        stitcher = cv2.Stitcher_create(cv2.Stitcher_SCANS)
        stitcher.setPanoConfidenceThresh(lowres_thresh)
        status, low_res_scan_preview = stitcher.stitch(low_res_images)
        low_res_integrated = count_integrated_images(stitcher) if status == cv2.Stitcher_OK else 0
        low_res_ratio = low_res_integrated / len(nef_paths)

        # Free up memory allocated to thumbnails immediately
        del low_res_images
        del low_res_scan_preview
        del stitcher
        gc.collect()
        log_memory("after low-res stitch")

        if status != cv2.Stitcher_OK:
            print(f"  [Error] Low-res stitching failed with error code: {status}")
            print("  Tip: Verify that the frame-to-frame step overlap is at least 20-30%.")
            if attempt_idx == len(attempt_thresholds):
                print(f"  Giving up after {number_of_attempts} attempts.")
            else:
                print("  Retrying with lower confidence thresholds...")
            continue

        print(f"  Low-res stitch included {low_res_integrated}/{len(nef_paths)} images ({low_res_ratio:.0%}).")
        if low_res_ratio < scan_min_inclusion_ratio:
            print(f"  [Warning] Low-res inclusion below {scan_min_inclusion_ratio:.0%}; retrying with lower thresholds.")
            if attempt_idx == len(attempt_thresholds):
                print(f"  Giving up after {number_of_attempts} attempts.")
            continue

        print("  Grid layout successfully calculated!")

        # Step 3: High resolution processing with safe RAM footprints
        print(" Step 3: Generating final high-res grid composition...")
        stitcher_high = cv2.Stitcher_create(cv2.Stitcher_SCANS)
        stitcher_high.setRegistrationResol(0.6)
        stitcher_high.setPanoConfidenceThresh(highres_thresh)
        high_res_images = []
        for path in nef_paths:
            with rawpy.imread(path) as raw:
                rgb = raw.postprocess(use_camera_wb=True, half_size=False)
            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            high_res_images.append(np.ascontiguousarray(bgr, dtype=np.uint8))
            del rgb, bgr

        log_memory("after high-res prep")
        print("  Assembling canvas matrices...")
        status, high_res_scan_result = stitcher_high.stitch(high_res_images)
        high_res_integrated = count_integrated_images(stitcher_high) if status == cv2.Stitcher_OK else 0
        high_res_ratio = high_res_integrated / len(nef_paths)

        # Instantly wipe the high-res list elements out of memory cache
        del high_res_images
        gc.collect()
        log_memory("after high-res stitch")

        if status == cv2.Stitcher_OK and high_res_ratio >= scan_min_inclusion_ratio:
            print(f"  High-res stitch included {high_res_integrated}/{len(nef_paths)} images ({high_res_ratio:.0%}).")
            output_suffix = "_FLAG" if attempt_idx > 1 else ""
            output_path = f"{output_dir}/{sample}_stitched_output{output_suffix}.jpg"
            cv2.imwrite(output_path, high_res_scan_result)
            print(f"  Success! Stitched grid saved to: {output_path}")

            del high_res_scan_result
            del stitcher_high
            gc.collect()
            log_memory("after output cleanup")
            stitched = True
            break

        print(f"  [Warning] High-res stitch only included {high_res_integrated}/{len(nef_paths)} images ({high_res_ratio:.0%}).")
        print("  Retrying with lower confidence thresholds...")
        if status == cv2.Stitcher_OK:
            del high_res_scan_result
        del stitcher_high
        gc.collect()
        log_memory("after failed output cleanup")

    if not stitched:
        print(f"  [Error] Could not achieve a stitch that included at least 90% of the sub-images after {number_of_attempts} attempts.")



Running images from: ALHIC2502-101-A
  [Memory] start sample: 83.1 MB
 Found 49 NEF files in folder 'ALHIC2502-101-A'.
 Attempt 1/6 with lowres=1.00, highres=1.00
 Step 1: Extracting low-res thumbnails to calculate alignment...
  [Memory] after low-res prep: 262.9 MB
 Step 2: Aligning 2D zig-zag grid structure...
  [Memory] after low-res stitch: 636.0 MB
  Low-res stitch included 49/49 images (100%).
  Grid layout successfully calculated!
 Step 3: Generating final high-res grid composition...
  [Memory] after high-res prep: 1054.5 MB
  Assembling canvas matrices...
  [Memory] after high-res stitch: 978.1 MB
  High-res stitch included 49/49 images (100%).
  Success! Stitched grid saved to: image_outputs//ALHIC2502-101-A_stitched_output.jpg
  [Memory] after output cleanup: 95.2 MB

Running images from: ALHIC2502-101-B
  [Memory] start sample: 95.2 MB
 Found 53 NEF files in folder 'ALHIC2502-101-B'.
 Attempt 1/6 with lowres=1.00, highres=1.00
 Step 1: Extracting low-res thumbnails to cal